# Notebook Detaille - Gestionnaire de Budget : Finance Zen
# (Version debutant - explications approfondies)

---

## C'est quoi ce projet ?

Ce projet c'est une **application web de gestion de budget personnel**. En gros, ca te permet de :
- Ajouter des depenses (courses, loyer, metro...) et des revenus (salaire, freelance...)
- Voir ton solde en temps reel (combien il te reste)
- Visualiser tes depenses/revenus dans un graphique en forme de donut
- Filtrer par categorie (alimentation, transport, loisirs...)
- Changer entre mode clair et mode sombre

### Pourquoi c'est un bon exercice pour apprendre ?

Parce qu'on touche a **tout** ce qu'un developpeur web junior doit savoir :

| Competence | Ce qu'on apprend ici |
|---|---|
| **Frontend** | React, TypeScript, composants, gestion d'etat avec useReducer |
| **Backend** | Creer une API REST avec Express.js |
| **Communication** | Faire parler le front et le back ensemble avec `fetch` |
| **Persistence** | Sauvegarder des donnees dans un fichier JSON |
| **Design** | CSS variables, theme sombre, animations, responsive |
| **Graphiques** | Afficher des donnees visuellement avec Chart.js |

---

## Avant de commencer : les prerequis

Si tu debutes, voila les notions que tu dois connaitre un minimum :

1. **HTML/CSS de base** : savoir ce qu'est une balise, un selecteur CSS, etc.
2. **JavaScript de base** : variables, fonctions, tableaux, objets, boucles
3. **Node.js** : avoir Node installe sur ton PC. C'est lui qui fait tourner le backend.
4. **Terminal** : savoir ouvrir un terminal et taper des commandes

> Si tu connais pas encore React ou TypeScript, pas de panique !
> Ce notebook explique chaque concept en detail au fur et a mesure.

---

# PARTIE 1 : L'architecture du projet (la structure)

## C'est quoi une architecture "client-serveur" ?

Imagine un restaurant :
- Le **client** (toi au restaurant) = le **navigateur web** (Chrome, Firefox...)
- Le **serveur** (le cuisinier en cuisine) = le **backend Express.js**
- Le **menu** que tu lis = le **frontend React** (l'interface)
- La **cuisine** ou on prepare = le **serveur** qui traite tes demandes
- Le **frigo** ou sont les ingredients = le **fichier JSON** (la base de donnees)

Quand tu cliques sur "Ajouter une transaction" dans l'interface :
1. Le **frontend** (React) prepare les donnees
2. Il envoie une **requete HTTP** au **backend** (Express)
3. Le **backend** recoit la requete, ecrit dans le fichier JSON
4. Il repond "OK, c'est fait" au frontend
5. Le **frontend** met a jour l'affichage

```
[TON NAVIGATEUR]                    [TON PC]

React (interface)                   Express (serveur)
   port 5173          ---HTTP-->       port 3001
                                         |
                                    lit/ecrit dans
                                  data/transactions.json
```

## C'est quoi un "port" ?

Un port c'est comme un numero de porte dans un immeuble. Ton PC a plein de ports.
- Le frontend ecoute sur la porte **5173** (c'est Vite qui choisit ce numero)
- Le backend ecoute sur la porte **3001** (c'est nous qui l'avons choisi)

## L'arborescence des fichiers

```
gestionnaire-budget/
|
|-- src/                        <-- FRONTEND (ce que l'utilisateur voit)
|   |-- App.tsx                 <-- Composant principal + types + logique d'etat
|   |-- index.css               <-- Tous les styles CSS
|   |-- main.tsx                <-- Point d'entree : "demarre React ici"
|   |-- components/             <-- Les composants React separes
|       |-- TransactionForm.tsx <-- Formulaire d'ajout de transaction
|       |-- BudgetChart.tsx     <-- Graphique donut (Chart.js)
|       |-- TransactionList.tsx <-- Historique + filtres par categorie
|
|-- data/
|   |-- transactions.json       <-- Notre "base de donnees" (un simple fichier)
|
|-- server.js                   <-- BACKEND (le serveur qui recoit les requetes)
|-- vite.config.ts              <-- Configuration de Vite
|-- package.json                <-- Liste des dependances et commandes
|-- index.html                  <-- La page HTML de base
```

### Pourquoi des composants separes ?

Au lieu de tout mettre dans un seul fichier `App.tsx` de 500 lignes, on a decoupe en **3 composants** :
- `TransactionForm` : le formulaire (gere ses propres champs)
- `BudgetChart` : le graphique (gere le canvas Chart.js)
- `TransactionList` : l'historique (gere le filtre par categorie)

Chaque composant est une "brique" reutilisable qui recoit des **props** (des parametres) du parent `App`.

---

# PARTIE 2 : `package.json` - La carte d'identite du projet

## C'est quoi package.json ?

C'est le fichier qui dit a Node.js : "voila ce dont j'ai besoin pour fonctionner".
C'est comme une liste de courses : tu listes toutes les librairies necessaires, et `npm install` les telecharge pour toi.

```json
{
  "name": "gestionnaire-budget",
  "type": "module",
  "scripts": {
    "dev": "vite",
    "server": "node server.js"
  },
  "dependencies": {
    "chart.js": "^4.4.7",
    "cors": "^2.8.6",
    "express": "^5.2.1",
    "react": "^19.2.0",
    "react-dom": "^19.2.0"
  }
}
```

### `"type": "module"`
Ca dit a Node.js d'utiliser les imports modernes (ESM) au lieu de `require()`.

### `"scripts"` - Les commandes raccourcis
- `npm run dev` = lance le frontend
- `npm run server` = lance le backend

### `"dependencies"` - Les librairies installees

| Librairie | A quoi elle sert |
|---|---|
| `react` | Creer des interfaces dynamiques |
| `react-dom` | Permet a React de manipuler le HTML |
| `express` | Creer un serveur HTTP facilement |
| `cors` | Autoriser les requetes entre differentes origines |
| `chart.js` | Dessiner des graphiques (ici un donut) |

---

## Comment lancer le projet

```bash
# Etape 1 : installer les dependances (a faire UNE SEULE FOIS)
npm install

# Etape 2 : ouvrir DEUX terminaux

# Terminal 1 : lancer le backend
npm run server
# -> "Serveur demarre sur http://localhost:3001"

# Terminal 2 : lancer le frontend
npm run dev
# -> "Local: http://localhost:5173"
```

> IMPORTANT : les deux terminaux doivent rester ouverts en meme temps !

---

# PARTIE 3 : `vite.config.ts` - Le proxy magique

**Vite** c'est un outil de dev pour le frontend. Il compile le TypeScript, recharge la page automatiquement, et sert les fichiers sur le port 5173.

```typescript
import { defineConfig } from 'vite'
import react from '@vitejs/plugin-react'

export default defineConfig({
    plugins: [react()],
    server: {
        proxy: {
            '/api': 'http://localhost:3001'
        }
    }
})
```

### Le proxy c'est quoi ?

Le navigateur bloque les requetes entre 2 ports differents (CORS). Le proxy redirige les URLs `/api` vers le backend sans que le navigateur s'en rende compte :
```
SANS proxy : fetch('http://localhost:3001/api/transactions') --> ERREUR CORS !
AVEC proxy : fetch('/api/transactions') --> Vite redirige vers port 3001 --> OK
```

---

# PARTIE 4 : `data/transactions.json` - La base de donnees

Un simple fichier JSON qui sert de base de donnees. Pas besoin d'installer PostgreSQL ou MongoDB.

```json
[
    {
        "id": "1772198316422yveub4k0zl",
        "name": "Courses supermarche",
        "amount": 85.50,
        "type": "depense",
        "category": "alimentation",
        "date": "2026-02-03"
    }
]
```

| Champ | Type | Explication |
|---|---|---|
| `id` | string | Identifiant unique (Date.now + Math.random) |
| `name` | string | Description de la transaction |
| `amount` | number | Montant en euros, toujours positif |
| `type` | string | `"revenu"` ou `"depense"` |
| `category` | string | Une des 8 categories |
| `date` | string | Date au format ISO (AAAA-MM-JJ) |

---

# PARTIE 5 : `server.js` - Le Backend Express

Le serveur tourne en permanence et attend des requetes pour y repondre.

## 5.1 - Les imports et la config
 
```javascript
import express from 'express'
import cors from 'cors'
import fs from 'fs'
import path from 'path'
import { fileURLToPath } from 'url'

const app = express()
const PORT = 3001

// en ESM y'a pas __dirname donc faut le reconstruir
const __filename = fileURLToPath(import.meta.url)
const __dirname = path.dirname(__filename)
const DATA_FILE = path.join(__dirname, 'data', 'transactions.json')

app.use(cors())
app.use(express.json())
```

## 5.2 - Les fonctions read/write

```javascript
function readTransactions() {
    try {
        const data = fs.readFileSync(DATA_FILE, 'utf-8')
        return JSON.parse(data)
    } catch (err) {
        return []
    }
}

function writeTransactions(transactions) {
    fs.writeFileSync(DATA_FILE, JSON.stringify(transactions, null, 4), 'utf-8')
}
```

## 5.3 - Les 4 routes de l'API REST

| Verbe | URL | Action |
|---|---|---|
| GET | `/api/transactions` | Recuperer tout |
| POST | `/api/transactions` | Ajouter |
| DELETE | `/api/transactions/:id` | Supprimer une |
| DELETE | `/api/transactions` | Tout effacer |

```javascript
app.get('/api/transactions', (req, res) => {
    res.json(readTransactions())
})

app.post('/api/transactions', (req, res) => {
    const transactions = readTransactions()
    transactions.unshift(req.body)
    writeTransactions(transactions)
    res.status(201).json(req.body)
})

app.delete('/api/transactions/:id', (req, res) => {
    const transactions = readTransactions()
    const filtered = transactions.filter(t => t.id !== req.params.id)
    if (filtered.length === transactions.length)
        return res.status(404).json({ error: 'Pas trouvee' })
    writeTransactions(filtered)
    res.json({ message: 'Supprimee' })
})

app.delete('/api/transactions', (req, res) => {
    writeTransactions([])
    res.json({ message: 'Tout supprime' })
})

app.listen(PORT, () => console.log(`Serveur sur http://localhost:${PORT}`))
```

---

# PARTIE 6 : `src/main.tsx` - Le demarrage de React

```typescript
import { StrictMode } from 'react'
import { createRoot } from 'react-dom/client'
import './index.css'
import App from './App.tsx'

createRoot(document.getElementById('root')!).render(
    <StrictMode>
        <App />
    </StrictMode>,
)
```

- `document.getElementById('root')` : React injecte toute l'app dans le `<div id="root">` de index.html
- Le `!` : dit a TypeScript "je sais que cet element existe"
- `<StrictMode>` : outil de dev qui detecte des bugs
- `<App />` : notre composant principal

---

# PARTIE 7 : `src/App.tsx` - Le composant principal

C'est le fichier central. Il contient les types, les constantes, le reducer, et le composant `App` qui assemble les 3 sous-composants.

## 7.1 - Les imports

```typescript
import { useReducer, useEffect, useState, useCallback } from 'react'
import TransactionForm from './components/TransactionForm'
import BudgetChart from './components/BudgetChart'
import TransactionList from './components/TransactionList'
```

### C'est quoi les hooks React ?

| Hook | En une phrase |
|---|---|
| `useState` | Memorise une valeur qui peut changer |
| `useReducer` | Comme useState mais pour des etats complexes |
| `useEffect` | Execute du code quand quelque chose change |
| `useCallback` | Memorise une fonction pour ne pas la recreer |

## 7.2 - Les types TypeScript

On utilise `type` (pas `interface`) pour definir la forme de nos donnees :

```typescript
export const CATEGORIES = [
    'alimentation', 'transport', 'loisirs', 'logement',
    'sante', 'salaire', 'freelance', 'autre',
] as const

export type Category = typeof CATEGORIES[number]
export type TransactionType = 'revenu' | 'depense'

export type Transaction = {
    id: string
    name: string
    amount: number
    type: TransactionType
    category: Category
    date: string
}
```

- `as const` : dit a TypeScript que le tableau est fixe (pas n'importe quel string)
- `export` : permet aux composants enfant d'importer ces types depuis `App.tsx`
- On utilise `import type` dans les composants car la config TypeScript l'exige pour les types

## 7.3 - Le Reducer

Le reducer c'est comme un distributeur automatique : tu dispatches une action, il retourne le nouvel etat.

```typescript
type BudgetAction =
    | { type: 'ADD_TRANSACTION';   payload: Transaction }
    | { type: 'DELETE_TRANSACTION'; payload: string }
    | { type: 'LOAD_TRANSACTIONS'; payload: Transaction[] }
    | { type: 'RESET_TRANSACTIONS' }

function budgetReducer(state, action) {
    switch (action.type) {
        case 'ADD_TRANSACTION':
            return { transactions: [action.payload, ...state.transactions] }
        case 'DELETE_TRANSACTION':
            return { transactions: state.transactions.filter(t => t.id !== action.payload) }
        case 'LOAD_TRANSACTIONS':
            return { transactions: action.payload }
        case 'RESET_TRANSACTIONS':
            return { transactions: [] }
    }
}
```

## 7.4 - Les handlers et le JSX

Les handlers font l'appel API puis dispatch l'action. Le JSX assemble les composants :

```tsx
return (
    <div id="app">
        <header> ... logo + bouton theme ... </header>
        <div className="balance-card"> ... solde ... </div>
        <div className="summary-row"> ... revenus + depenses ... </div>

        <div className="main-grid">
            <TransactionForm onSubmit={handleAddTransaction} />
            <BudgetChart transactions={state.transactions} theme={theme} />
        </div>

        <TransactionList
            transactions={state.transactions}
            onDelete={handleDelete}
            onReset={handleReset}
        />
    </div>
)
```

Chaque composant recoit des **props** (donnees ou callbacks). Le parent garde le controle sur l'etat global et les appels API.

---

# PARTIE 8 : Les composants

## 8.1 - `TransactionForm.tsx`

Gere ses propres champs en interne. Recoit `onSubmit` du parent pour remonter les donnees.

```typescript
import { useState } from 'react'
import { CATEGORIES } from '../App'
import type { Category, TransactionType } from '../App'

type FormProps = {
    onSubmit: (name: string, amount: number, type: TransactionType, category: Category) => void
}

function TransactionForm({ onSubmit }: FormProps) {
    const [name, setName] = useState('')
    const [amount, setAmount] = useState('')
    const [type, setType] = useState<TransactionType>('depense')
    const [category, setCategory] = useState<Category>('alimentation')

    function handleSubmit(e: React.FormEvent) {
        e.preventDefault()
        if (!name.trim() || parseFloat(amount) <= 0) return
        onSubmit(name.trim(), parseFloat(amount), type, category)
        setName('')
        setAmount('')
    }
    // ... le JSX du formulaire ...
}
```

Le flux :
```
[App] --onSubmit={handleAddTransaction}--> [TransactionForm]
                                              utilisateur clique Ajouter
                                              appelle onSubmit(name, amount...)
[App] <-- handleAddTransaction s'execute
  --> fetch POST + dispatch
```

## 8.2 - `BudgetChart.tsx`

Recoit les transactions et le theme. Gere le toggle depenses/revenus et le cycle de vie du chart.

```typescript
type ChartProps = {
    transactions: Transaction[]
    theme: 'light' | 'dark'
}

function BudgetChart({ transactions, theme }: ChartProps) {
    const chartRef = useRef<HTMLCanvasElement>(null)      // le canvas
    const chartInstance = useRef<Chart | null>(null)      // l'objet Chart.js
    // useEffect detruit et recree le chart quand les donnees changent
}
```

`useRef` c'est un tiroir : on stocke l'objet Chart.js sans declencher de re-render.

## 8.3 - `TransactionList.tsx`

Recoit les transactions + fonctions de suppression/reset. Le filtre par categorie est en interne.

```typescript
type ListProps = {
    transactions: Transaction[]
    onDelete: (id: string) => void
    onReset: () => void
}

function TransactionList({ transactions, onDelete, onReset }: ListProps) {
    const [filter, setFilter] = useState<Category | 'tout'>('tout')
    const filtered = filter === 'tout'
        ? transactions
        : transactions.filter(t => t.category === filter)
    // ... le JSX avec les boutons de filtre et la liste ...
}
```

---

# PARTIE 9 : `src/index.css` - Le Design

## Variables CSS et theme

```css
:root {
    --bg-primary: #f5efe6;
    --text-primary: #3e2c1c;
    --accent: #8b6f47;
    --income-color: #2e8b57;
    --expense-color: #c0392b;
}

[data-theme="dark"] {
    --bg-primary: #1a1410;
    --text-primary: #e8dfd5;
}
```

Quand React met `data-theme="dark"` sur `<html>`, les variables sont ecrasees et tout le design change.

## Glassmorphisme

```css
.panel {
    background: rgba(210, 190, 165, 0.35);
    backdrop-filter: blur(12px);
    border-radius: 14px;
}
```

## Responsive

```css
.main-grid { grid-template-columns: 1fr 1fr; }    /* 2 colonnes */
@media (max-width: 700px) {
    .main-grid { grid-template-columns: 1fr; }    /* 1 colonne sur mobile */
}
```

## Canvas du graphique

On laisse Chart.js gerer la taille du canvas, sinon le rendu devient flou :
```css
.chart-wrapper { max-width: 280px; position: relative; }
/* PAS de width/height !important sur le canvas ! */
```

---

# PARTIE 10 : Resume

## Le flux complet d'ajout d'une transaction

```
1. L'utilisateur remplit le formulaire dans TransactionForm
2. Il clique "Ajouter" -> handleSubmit()
3. TransactionForm appelle la prop onSubmit(name, amount, type, category)
4. App recoit l'appel dans handleAddTransaction()
5. fetch() envoie un POST au backend
6. Vite redirige vers Express port 3001
7. Express ecrit dans transactions.json
8. dispatch() met a jour le state React
9. React re-render les composants
```

## Tableau de reference

| Fichier | Role |
|---|---|
| `package.json` | Dependances + scripts |
| `vite.config.ts` | Config Vite + proxy |
| `server.js` | API REST backend |
| `data/transactions.json` | Base de donnees |
| `src/main.tsx` | Point d'entree React |
| `src/App.tsx` | Composant principal + types + reducer |
| `src/components/TransactionForm.tsx` | Formulaire |
| `src/components/BudgetChart.tsx` | Graphique donut |
| `src/components/TransactionList.tsx` | Historique + filtres |
| `src/index.css` | Styles CSS |

## Commandes utiles

```bash
npm install          # installer les dependances
npm run server       # lancer le backend (port 3001)
npm run dev          # lancer le frontend (port 5173)
```